In [2]:
import sys
sys.path.append('../')

import numpy as np
import pandas as pd
import importlib
import matplotlib.pyplot as plt

import env.trading_env_v3 as te3
import features.feature_engineering as fe
importlib.reload(te3)
importlib.reload(fe)

from stable_baselines3 import PPO, SAC, A2C
from stable_baselines3.common.callbacks import BaseCallback


df = pd.read_excel('../data/data.xlsx', skiprows=6, header=0)
df.columns = ['date', 'gold', 'silver', 'copper']
df = df[pd.to_datetime(df['date'], errors='coerce').notna()]
df['date'] = pd.to_datetime(df['date'])
df.set_index('date', inplace=True)


features_final = fe.build_features(df)


train = features_final[features_final.index <= '2023-12-31']
test = features_final[features_final.index >= '2024-01-01']
price_train = df[df.index.isin(train.index)]
price_test = df[df.index.isin(test.index)]

print(f"Train: {len(train)}days, Test: {len(test)}days")

Train: 2592days, Test: 485days


In [4]:
class RewardCallback(BaseCallback):
    def __init__(self):
        super().__init__()
        self.episode_rewards = []
        self.current_rewards = []
    
    def _on_step(self):
        reward = self.locals['rewards'][0]
        self.current_rewards.append(reward)
        
        if self.locals['dones'][0]:
            self.episode_rewards.append(sum(self.current_rewards))
            self.current_rewards = []
        return True

def train_agent(algorithm, env, total_timesteps=1_000_000, **kwargs):
    callback = RewardCallback()
    model = algorithm("MlpPolicy", env, verbose=0, **kwargs)
    model.learn(total_timesteps=total_timesteps, callback=callback)
    return model, callback

# PPO
print("Training PPO...")
train_env = te3.MetalTradingEnvV3(train, price_train)
ppo_sharpe, ppo_cb = train_agent(PPO, train_env, learning_rate=3e-4, n_steps=2048, batch_size=64, n_epochs=10, gamma=0.99)
ppo_sharpe.save("../agents/ppo_sharpe_1m")
np.save("../agents/ppo_sharpe_rewards_1m.npy", ppo_cb.episode_rewards)
print(f"PPO done. Average for the last 10 episodes: {np.mean(ppo_cb.episode_rewards[-10:]):.4f}")

# SAC
print("Training SAC...")
train_env = te3.MetalTradingEnvV3(train, price_train)
sac_sharpe, sac_cb = train_agent(SAC, train_env, learning_rate=3e-4, batch_size=256, buffer_size=100_000, learning_starts=1000, gamma=0.99)
sac_sharpe.save("../agents/sac_sharpe_1m")
np.save("../agents/sac_sharpe_rewards_1m.npy", sac_cb.episode_rewards)
print(f"SAC done. Average for the last 10 episodes: {np.mean(sac_cb.episode_rewards[-10:]):.4f}")

# A2C
print("Training A2C...")
train_env = te3.MetalTradingEnvV3(train, price_train)
a2c_sharpe, a2c_cb = train_agent(A2C, train_env, learning_rate=7e-4, n_steps=5, gamma=0.99)
a2c_sharpe.save("../agents/a2c_sharpe_1m")
np.save("../agents/a2c_sharpe_rewards_1m.npy", a2c_cb.episode_rewards)
print(f"A2C done. Average for the last 10 episodes: {np.mean(a2c_cb.episode_rewards[-10:]):.4f}")

print("\nAll training completed!")

Training PPO...


KeyboardInterrupt: 